In [ ]:
#!pip install -q lxml
#!pip install unidecode
import pandas as pd
import numpy as np
#import geopandas as gpd
import urllib#pour récupérer les données
import bs4#pour rendre lisibles les données
import lxml
import re
import time
from unidecode import unidecode
import urllib
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

from urllib import request

In [ ]:
# diaporama anglais pas se limiter 
# partie méthodo conclusion
# refaire en anglais

In [ ]:
"""carte avec expériences précédentes en fonction du domaine AR 2
type de trajectoire révolution : investissement et trajectoire  
carte

montrer par domaine l'expérience internationale, au moment de leur nomination (sans étranger), graphe 

regarder expérience internationale dans les 3 ans (à partir de +1) après leur nomination
-> retraite / parisianisation ou pas ?

qq type dessus

paris étranger avec frontière actuelle sous terreur (93, 96) par domaine d'activité (noblesse ?)
prendre le plus lointain quand plusieurs ?

expérience internationale après ?"""

In [ ]:
#senateur = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/test_17_senateurs.csv")
senateur = pd.read_excel("C:/Users/sylva/OneDrive/Bureau/senat/Data/Prosopographie sénateurs.xlsx")


In [ ]:
senateur = senateur.dropna(how = 'all')

In [ ]:
years = list(range(1789, 1816))

In [ ]:
senateur["naiss"].apply(str)

In [ ]:
def to_date_naiss(x):
    if x == x:
        return float("17"+str(x)[-2:])
    else:
        return None
def to_date_nomin(x):
    if x == x:
        if float(str(x)[2:4])>50:
          return float("17"+str(x)[2:4])
        else:
          return float("18"+str(x)[2:4])
    else:
        return None
def to_date_deces(x):
    if x == x:
        return float("18"+str(x)[2:4])
    else:
        return None

In [ ]:
senateur["date nomination"]

In [ ]:
senateur["annee naiss"] = senateur["naiss"].apply(to_date_naiss)
senateur["annee nomin"] = senateur["date nomination"].apply(to_date_nomin)
senateur["annee deces"] = senateur["date mort"].apply(to_date_deces)

In [ ]:
senateur[["annee naiss", "annee nomin", "annee deces"]]

In [ ]:
def to_positions_princ(x):
    if pd.isna(x):
        return ''
    elif x.startswith('admin') or x == 'enfance':
        return 'Civil service'
    elif x in ['armée', 'haute armée'] or x.startswith('armee'):
        return 'Military'
    elif x in ['paysan', 'artisan', 'cultivateur']:
        return 'Business'
    elif x.startswith('médec') or x.startswith('medec'):
        return 'Civil service'
    elif x.startswith('clerge'):
        return 'Clergy'
    elif x in ['negoce finance', 'agriculture industrie']:
        return 'Business'
    elif x in ['juridique']:
        return 'Judiciary'
    elif x in ['savoir culture']:
        return 'Knowledge Culture'
    elif x in ['notabilite']:
        return 'Local personality'
    elif x in ['ambassadeur']:
        return 'Diplomacy'
    else:
        return x

In [ ]:
senateur['position'] = senateur['domaine act AR 2'].apply(to_positions_princ)

In [ ]:
senateur['position'].unique()

In [ ]:
def set_place_at_year(personne, year=1800):
    if year < 1789:
        period = "AR"
        nb_period_max = 2
    if 1788 < year < 1799:
        period = "revolution"
        nb_period_max = 10
    if 1798 < year < 1805:
        period = "consulat"
        nb_period_max = 5
    if 1804 < year < 1816:
        period = "empire"
        nb_period_max = 7
    for num_period in range(1, nb_period_max):
        date_num = personne["date "+period+" "+str(num_period)]
        if date_num == str(date_num):
            if str(year) in [str(date) for date in date_num.replace("/", "-").split("-")]:
                date_num = year
            else:
                date_num = 0
        if pd.notna(date_num) and int(date_num)==int(year):
            lieu = personne["lieu "+period+" "+str(num_period)]
            if (type(lieu)==str) : #& (lieu != "paris")
                return personne["lieu "+period+" "+str(num_period)].title().rstrip()
    return ""

In [ ]:
def set_activity_at_year(personne, year=1800):
    if year < 1789:
        period = "AR"
        nb_period_max = 2
    if 1788 < year < 1799:
        period = "revolution"
        nb_period_max = 10
    if 1798 < year < 1805:
        period = "consulat"
        nb_period_max = 5
    if 1804 < year < 1816:
        period = "empire"
        nb_period_max = 7
    for num_period in range(1, nb_period_max):
        date_num = personne["date "+period+" "+str(num_period)]
        if date_num == str(date_num):
            if str(year) in [str(date) for date in date_num.replace("/", "-").split("-")]:
                date_num = year
            else:
                date_num = 0
        if pd.notna(date_num) and int(date_num)==int(year):
            act = personne[period+" "+str(num_period)]
            if (type(act)==str) : #& (lieu != "paris")
                return personne[period+" "+str(num_period)].title().rstrip()
    return ""

In [ ]:
def keep_place_at_year(personne, year):
    if personne["nom_local"+str(year)] != '':
        return personne["nom_local"+str(year)]
    elif year <= personne['annee deces']:
        lieu_actuel = personne['ville naissance']
        for annee in range(1750,year):
            if annee in personne.index:
                if personne["nom_local"+str(annee)]!='':
                    lieu_actuel = personne["nom_local"+str(annee)]
        return lieu_actuel
    else:
        return ''

In [ ]:
def keep_activity_at_year(personne, year):
    if personne["activite"+str(year)] != '':
        return personne["activite"+str(year)]
    elif year <= personne['annee deces']:
        activity_actuel = personne['position']
        for annee in range(1750,year):
            if annee in personne.index:
                if personne["activite"+str(annee)]!='':
                    activity_actuel = personne["activite"+str(annee)]
        return activity_actuel
    else:
        return ''

In [ ]:
for year in years:
    senateur["nom_local"+str(year)] = senateur.apply(set_place_at_year, axis = 1, args = [year])
    senateur["nom_local"+str(year)] = senateur.apply(keep_place_at_year, axis = 1, args = [year])
    senateur["activite"+str(year)] = senateur.apply(set_activity_at_year, axis = 1, args = [year])
    senateur["activite"+str(year)] = senateur.apply(keep_activity_at_year, axis = 1, args = [year])

In [ ]:
def to_nomin(x):
    if pd.isna(x):
        return ''
    elif x == 'sur titre':
        return 'On title'
    elif x =='coopt. restr.':
        return 'Restricted cooptation'
    elif x =='coopt. part.' or x == 'coopt. libre':
        return 'Free cooptation'
    elif x =='désign. libre':
        return 'Free designation'
    else:
        return x

In [ ]:
senateur['type nomination'] = senateur['type nomination'].apply(to_nomin)

In [ ]:
def to_activ_princ(x):
    if pd.isna(x):
        return ''
    elif x.startswith('ambassadeur') or x in ['Diplomatie', 'diplomatie']:
        return 'Diplomacy'
    elif x =='savoir culture' or x in ['Savoir Culture', 'sciences politique', 'sciences', 'art']:
        return 'Knowledge culture'
    elif x == 'militaire' or x=='Armee' or x == 'military' or x in ['Arme', 'armée']:
        return 'Military'
    elif x == 'clerge' or x in ['clergy', 'clergé'] or x.startswith('clerge') or x=='Clerge':
        return 'Clergy'
    elif x == 'juridique' or x=='Magistrature' or x in['Cour', 'juridique politique']:
        return 'Judiciary'
    elif x == 'notabilite' or x == 'Noabilité' or x == 'Notabilite':
        return 'Local Personality'
    elif x == 'Banque De France' or x == 'admin' or x.startswith('Admin') or x == 'enfance':
        return 'Civil service'
    elif x in ['Mairie', 'Assemblée Locale', 'Gouvernement', 'Parlement', "Conseil D'Etat", 'politique', 
               'Prefecture', 'Politique', 'Sénateur', 'Assemblee Locale', 'Commissaire Extraordinaire']:
        return 'Politics'
    elif x == 'Voyage':
        return 'Travel'
    elif x == 'Prison':
        return 'Inmate'
    elif x in ['Retraite', 'Perte Fonction', 'Conflit Disgrace', 'Exil', 'retraite']:
        return 'Retirement'
    elif x in ['business', 'Negoce Commerce', 'Agriculture Industrie', 'négoce finance', 'Negoce Finance', 'agriculture industrie']:
        return 'Business'
    else:
        return x

In [ ]:
for year in years:
    senateur["activite"+str(year)] = senateur["activite"+str(year)].apply(to_activ_princ)
    if year > 1798:
        print(senateur["activite"+str(year)].unique())

In [ ]:
senateur

In [ ]:
#senateur["position_sociale"]=senateur["1. place hiérarchie sociale famille"]

In [ ]:
#senateur["position_sociale"].unique()

In [ ]:
#senateur["position_sociale_restreint"] = senateur["position_sociale"].apply(to_positions_princ)

In [ ]:

senateur['jours conges 1800'] = senateur['jours conges (1799-)1800']
senateur['role 1799-1802'] = senateur['secretariat 1799-1802'].fillna(0) + senateur['president 1799-1802'].fillna(0)
senateur['role 1802-1804'] = senateur['secretariat 1802-1804'].fillna(0) + senateur['president 1802-1804'].fillna(0)
senateur['role 1804-1814'] = senateur['secretariat 1804-1814'].fillna(0) + senateur['president 1804-1814'].fillna(0)
for year in years:
    if year>1798 and year<1802:
        senateur['role' + str(year)] = senateur['role 1799-1802']
    elif year == 1802:
        senateur['role' + str(year)] = senateur['role 1799-1802'] + senateur['role 1802-1804']
    elif year == 1803:
        senateur['role' + str(year)] = senateur['role 1802-1804']
    elif year == 1804:
        senateur['role' + str(year)] = senateur['role 1802-1804'] + senateur['role 1804-1814']
    elif year > 1804:
        senateur['role' + str(year)] = senateur['role 1804-1814']

In [ ]:
senateur['previous participation to assembly'] = senateur['nb pres. ass. avant nomin'].fillna(0) + senateur['nb secretaire assemblee avant nomin'].fillna(0)
senateur['other participation to assembly'] = senateur['nb pres. ass. consulat-empire'].fillna(0) + senateur['nb secrétaire assemblee consulat-empire'].fillna(0)
senateur['senatorerie'] = senateur['[tranformer en lieu] sénatorerie']
senateur['revolution trajectory'] = senateur['trajectoire dominante révolution 1']

In [ ]:
senateur['revolution trajectory'] = senateur['revolution trajectory'].apply(to_activ_princ)

In [ ]:
senateur['revolution trajectory'].unique()

In [ ]:
def get_oui(x):
    if x == 'oui ':
        return 'oui'
    else:
        return x
senateur['noblesse AR'] = senateur['noblesse AR'].apply(get_oui)

In [ ]:
senateur_for_map = senateur[["nom_local"+str(year) for year in years]+ ['lieu AR 1', 'lieu AR 2'] +
                             ["activite"+str(year) for year in years] +
                             ["annee nomin", "nom", "annee naiss", "ville naissance", "annee deces", 'senatoreries', 
                            'senatorerie', 'revolution trajectory',
                              'jours conges 1800', 'jours conges 1801', 'jours conges 1802', 'jours conges 1803',
                            'nationalite', 'position', 'other participation to assembly',
                             'previous participation to assembly', 'noblesse AR'] + 
                            ['role' + str(year) for year in range(1799, 1816)]]

In [ ]:
Noms_speciaux = {"Allemagne":"Berlin", 'Étranger': "Paris", 'Provence': 'Aix-en-Provence',
                "Sur Le Mein": "Sur le Main", "Sambre Et Mesuse": "Namur", "Sambre Et Meuse": "Namur",
                "Aix": 'Aix-en-Provence', 'Campagne': 'Paris', 'Cassel (All)': 'Cassel, Allemagne',
                'Austerlitz' : 'Brno', 'Frnace' : 'France', 'Eylau': 'Bagrationovsk', 'Ratisbone':'Ratisbonne',
                'Alsace' : 'Strasbourg', 'Ouest': 'Ouest de la France', 'Languedoc':'Toulouse',
                'Uk': 'Angleterre', 'Midi':'Sud de la France', 'Marengo' : 'Alexandrie (Piémont)',
                 'Lorraine':'Metz'
                }
#Df_special = pd.DataFrame.from_dict(Noms_speciaux, orient = "index").reset_index()
#Df_special.columns = ["Nom en francais", "Nom local"]


In [ ]:
Nationalites = {'piémontais': 'Italian', 'italie':'Italian', 'Italie':'Italian', 'savoyard':'Italian',
               'gênois':'Italian', 'romain':'Italian', 'florentin':'Italian', 'toscan':'Italian',
                'Autriche': 'Austrian', 'hollande':'Dutch',
               'allemagne':'German','allemand':'German', 'bruxellois':'Belgian', 'belge':'Belgian', 'français': 'French'}

In [ ]:
def change_by_dic(x, dic):
    if x in dic.keys():
        return dic[x]
    else:
        return x
senateur_for_map["nation_short"] = senateur_for_map['nationalite'].apply(change_by_dic, args = [Nationalites])
for year in years:
    senateur_for_map["nom_local"+str(year)] = senateur_for_map["nom_local"+str(year)].apply(change_by_dic, args = [Noms_speciaux])

In [ ]:
senateur_for_map.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")

In [ ]:
for col in senateur_for_map:
    if "Zürich" in senateur_for_map[col].to_list():
        print(col)

In [ ]:
senateur_for_map.columns

In [ ]:
senateur_for_map['noblesse AR'].unique()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

In [ ]:
def to_coopt(x):
    if x == 'coopt. part.' or x == 'coopt. libre':
        return 'coopt. libre'
    else:
        return x
senateur['type nomination'] = senateur['type nomination'].apply(to_coopt)

In [ ]:
df = pd.DataFrame(columns = ['year', 'number', 'type nomination'])
for type_nomination in senateur['type nomination'].unique():
    df_type = senateur[senateur['type nomination']==type_nomination]
    for year in range(1799, 1815):
        df_year = df_type[df_type['annee nomin']==year]
        df_temp = pd.DataFrame.from_dict({'number': [len(df_year)], 'year': [year], 'type nomination':[type_nomination]}, orient = 'columns')
        df = pd.concat([df, df_temp], axis = 0)
        

In [ ]:
senateur['type nomination'].unique()

In [ ]:
path_to_save = 'C:/Users/sylva/OneDrive/Bureau/senat/plot/'


In [ ]:
Fig = px.line(df, x= "year", y = "number", color = 'type nomination', 
                   labels = {"number": "number of new senators"})#, color = colname)
Fig.update_layout(plot_bgcolor='white')

In [ ]:
Fig.write_image(path_to_save + 'type nomination.png')